# Phase 1 Supervised Fine-Tuning Colab Lab - Qwen 0.5B and Gemma 3 1B

This notebook has two jobs:

1. **Lab path:** a visible 50-step training loop you can inspect cell by cell. Use this when you want to understand what is happening, test T4 memory, or switch between Qwen 0.5B and Gemma 3 1B.
2. **Production path:** the one-command Trainer CLI used by the repo configs. Use this once the lab path looks healthy.

The original Colab run proved two important things that this notebook keeps explicit:

- Qwen 0.5B can run a short T4 canary.
- Gemma 3 1B can load and train on T4 only when we use the memory recipe: batch size 1, gradient accumulation, gradient checkpointing, and `bitsandbytes.optim.AdamW8bit`.

Start with the lab path. When it works, run the production CLI cell.

## Step 1 - verify the GPU

If `nvidia-smi` errors or shows no GPU, the runtime is not set to T4. Stop here and switch it: Runtime -> Change runtime type -> T4 GPU.

In [ ]:
!nvidia-smi

## Step 2 - clone the repo and install dependencies

Editable install keeps the notebook pointed at the repo source. The explicit `sys.path` cell below handles Colab Python 3.12 editable-install quirks.

In [ ]:
!git clone https://github.com/shannan-liu1/finpost.git
%cd finpost
!pip install -q -e .

## Step 3 - imports and Colab path fix

This mirrors the original notebook's first useful sanity check: confirm that Python imports `finpost` from `/content/finpost/src`, then confirm CUDA from inside PyTorch.

In [ ]:
from __future__ import annotations

import gc
import importlib
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

REPO_ROOT = '/content/finpost'
SRC_PATH = os.path.join(REPO_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

if 'finpost' in sys.modules:
    importlib.reload(sys.modules['finpost'])

import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from finpost.training.config import Config, DataConfig, ModelConfig, PackingConfig, TrainingConfig
from finpost.training.dataset import make_loaders
from finpost.training.optim import build_lr_scheduler, build_optimizer
from finpost.training.sft import compute_masked_ce_loss

print('finpost import path:', os.path.join(SRC_PATH, 'finpost'))
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## Step 4 - choose the run profile

Change only `RUN_PRESET` for the common paths:

- `qwen_500m`: the Phase 1 default model. Use this first.
- `gemma_1b_t4`: the larger stress test that matched the manual Colab work. Requires a Hugging Face token if the model is gated for your account.

The lab path defaults to GSM8K only because it is the fastest way to see whether the model trains. The production CLI later uses the checked-in YAML for the combined GSM8K + MATH run.

In [ ]:
RUN_PRESET = 'qwen_500m'      # 'qwen_500m' or 'gemma_1b_t4'
NUM_STEPS = 50                  # 50 is enough to prove the loop and see a loss trace
DATA_SOURCES = ['gsm8k']         # fast lab path; production config uses gsm8k + math

PRESETS = {
    'qwen_500m': {
        'model_id': 'Qwen/Qwen2.5-0.5B',
        'batch_size': 4,
        'grad_accum_steps': 1,
        'max_seq_len': 1024,
        'lr': 1e-4,
        'dtype': 'bfloat16',
        'use_safetensors': True,
        'gradient_checkpointing': False,
        'use_8bit_adamw': False,
        'hf_token_secret': None,
        'checkpoint_dir': '/content/qwen2_5_0_5b_checkpoint_50',
    },
    'gemma_1b_t4': {
        'model_id': 'google/gemma-3-1b-it',
        'batch_size': 1,
        'grad_accum_steps': 4,
        'max_seq_len': 1024,
        'lr': 1e-4,
        'dtype': 'bfloat16',
        'use_safetensors': True,
        'gradient_checkpointing': True,
        'use_8bit_adamw': True,
        'hf_token_secret': 'FINPOST_LOCAL_GEMMA',
        'checkpoint_dir': '/content/gemma3_1b_checkpoint_50',
    },
}

profile = PRESETS[RUN_PRESET]
print('Run profile')
for key, value in profile.items():
    print(f'  {key}: {value}')
print('  data_sources:', DATA_SOURCES)
print('  num_steps:', NUM_STEPS)
print('  effective_batch:', profile['batch_size'] * profile['grad_accum_steps'])

## Step 5 - build the training config

This uses the same Pydantic config objects as the production trainer, but keeps the knobs visible in the notebook. That is the main usability difference from the pure CLI path.

In [ ]:
warmup_steps = max(1, min(NUM_STEPS // 10, NUM_STEPS - 1))


config = Config(
    model=ModelConfig(
        base_model_id=profile['model_id'],
        dtype=profile['dtype'],
        use_safetensors=profile['use_safetensors'],
    ),
    data=DataConfig(
        sources=DATA_SOURCES,
        val_split_pct=5.0,
        seed=42,
    ),
    training=TrainingConfig(
        max_steps=NUM_STEPS,
        warmup_steps=warmup_steps,
        lr=profile['lr'],
        weight_decay=0.01,
        grad_accum_steps=profile['grad_accum_steps'],
        grad_clip=1.0,
        val_every_n_steps=max(10, NUM_STEPS // 2),
        checkpoint_every_n_steps=max(25, NUM_STEPS),
        per_device_batch_size=profile['batch_size'],
    ),
    packing=PackingConfig(
        max_seq_len=profile['max_seq_len'],
        isolate_documents=True,
    ),
)

print('Config built')
print('  model:', config.model.base_model_id)
print('  dtype:', config.model.dtype)
print('  sources:', config.data.sources)
print('  batch:', config.training.per_device_batch_size)
print('  grad_accum_steps:', config.training.grad_accum_steps)
print('  max_seq_len:', config.packing.max_seq_len)
print('  lr:', config.training.lr)

## Step 6 - install optional memory dependency

Gemma 3 1B on T4 needs 8-bit AdamW. Qwen 0.5B does not.

In [ ]:
if profile['use_8bit_adamw']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'], check=True)
    print('bitsandbytes installed')
else:
    print('bitsandbytes not needed for this profile')

## Step 7 - load tokenizer and model

This cell prints memory after load. For the prior Gemma T4 run, model load landed around 2 GB allocated before optimizer state and activations.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('This notebook is intended for a Colab T4 GPU runtime.')

secret_name = profile['hf_token_secret']
if secret_name is not None:
    try:
        from google.colab import userdata
        from huggingface_hub import login

        token = userdata.get(secret_name)
        if token:
            login(token=token)
            print(f'Logged in to Hugging Face using Colab secret {secret_name!r}')
        else:
            print(f'No Colab secret named {secret_name!r}; public models may still load, gated models will fail.')
    except Exception as exc:
        print('Hugging Face login skipped:', repr(exc))

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(config.model.base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('  pad_token_id =', tokenizer.pad_token_id)
print('  eos_token_id =', tokenizer.eos_token_id)

print('Loading model...')
dtype = getattr(torch, config.model.dtype)
model = AutoModelForCausalLM.from_pretrained(
    config.model.base_model_id,
    dtype=dtype,
    use_safetensors=config.model.use_safetensors,
).to(device)

if profile['gradient_checkpointing']:
    model.gradient_checkpointing_enable()
    if hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
    print('Gradient checkpointing enabled')

model.train()
print('GPU memory allocated after model load GB:', round(torch.cuda.memory_allocated() / 1e9, 2))

## Step 8 - build loaders and inspect the first batch

This is the fastest way to catch bad packing, masking, or sequence length settings before spending time in the training loop.

In [ ]:
print('Building data loaders...')
train_loader, val_loader = make_loaders(config, tokenizer)
print('Train examples:', len(train_loader.dataset))
print('Val examples:', len(val_loader.dataset))

peek = next(iter(train_loader))
print('First batch shapes')
print('  input_ids:', tuple(peek['input_ids'].shape))
print('  labels:', tuple(peek['labels'].shape))
print('  attention_mask:', tuple(peek['attention_mask'].shape))
print('  position_ids:', tuple(peek['position_ids'].shape))
print('  response tokens:', int((peek['labels'] != -100).sum().item()))

## Step 9 - build optimizer and scheduler

The default uses the repo's AdamW factory. The Gemma T4 profile swaps in `bitsandbytes.optim.AdamW8bit` to keep optimizer state small enough for free Colab.

In [ ]:
if profile['use_8bit_adamw']:
    import bitsandbytes as bnb

    optimizer = bnb.optim.AdamW8bit(
        model.parameters(),
        lr=config.training.lr,
        weight_decay=config.training.weight_decay,
    )
else:
    optimizer = build_optimizer(
        model,
        lr=config.training.lr,
        weight_decay=config.training.weight_decay,
    )

scheduler = build_lr_scheduler(
    optimizer,
    total_steps=config.training.max_steps,
    warmup_steps=config.training.warmup_steps,
)
print('Optimizer:', type(optimizer).__name__)
print('Initial lr:', optimizer.param_groups[0]['lr'])

## Step 10 - run the visible 50-step lab loop

This is intentionally explicit. It uses the same core pieces as the production trainer: packed batches, masked cross-entropy, gradient accumulation, gradient clipping, and the warmup/cosine scheduler.

In [ ]:
losses: list[float] = []
loader_iter = iter(train_loader)
grad_accum = config.training.grad_accum_steps

print(f'Running {NUM_STEPS} optimizer steps')
for step in range(1, NUM_STEPS + 1):
    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    response_tokens = 0
    rows = 0
    seq_len = 0

    for _ in range(grad_accum):
        try:
            batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            batch = next(loader_iter)

        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device).bool()
        position_ids = batch['position_ids'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
        )
        loss = compute_masked_ce_loss(outputs.logits, labels)
        (loss / grad_accum).backward()

        step_loss += float(loss.detach().float().item()) / grad_accum
        response_tokens += int((batch['labels'] != -100).sum().item())
        rows, seq_len = batch['input_ids'].shape

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.training.grad_clip)
    optimizer.step()
    scheduler.step()

    losses.append(step_loss)
    print(
        f'step {step:03d}: loss={step_loss:.4f} '
        f'lr={optimizer.param_groups[0]["lr"]:.2e} '
        f'grad_norm={float(grad_norm):.2f} '
        f'shape=({rows}x{seq_len}) resp_tokens={response_tokens}'
    )

print()
if any(not torch.isfinite(torch.tensor(v)).item() for v in losses):
    print('FAIL: NaN or inf in loss curve')
elif len(losses) > 1 and losses[-1] < losses[0]:
    print(f'OK: loss {losses[0]:.4f} -> {losses[-1]:.4f} (drop {losses[0] - losses[-1]:.4f})')
else:
    print('WARN: loss did not decrease over this short run; inspect the curve and try more steps or a lower LR.')

print('GPU memory allocated after training GB:', round(torch.cuda.memory_allocated() / 1e9, 2))

## Step 11 - plot the loss curve

The plot is a quick visual sanity check. On a 50-step run, noise is normal; you are looking for finite losses and a broadly downward trend.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, marker='o', markersize=3, linewidth=1)
plt.xlabel('optimizer step')
plt.ylabel('masked cross-entropy loss')
plt.title(f'{config.model.base_model_id} on {"+".join(config.data.sources)} - {len(losses)} steps')
plt.grid(alpha=0.3)
plt.show()

## Step 12 - save the lab checkpoint

This is the lightweight Hugging Face checkpoint path from the original notebook. The production CLI uses the repo's atomic checkpoint format; this lab checkpoint is for quick Colab experimentation and reloading with `from_pretrained`.

In [ ]:
CHECKPOINT_DIR = profile['checkpoint_dir']
model.save_pretrained(CHECKPOINT_DIR, safe_serialization=True)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print('Saved to', CHECKPOINT_DIR)

## Step 13 - optional: clear memory, reload the checkpoint, continue

The original notebook hit an out-of-memory error when reloading Gemma while the old model was still occupying VRAM. This cell frees the old objects first.

In [ ]:
# Run this cell only if you want to prove the saved lab checkpoint reloads.
del model
if 'optimizer' in globals():
    del optimizer
if 'scheduler' in globals():
    del scheduler
gc.collect()
torch.cuda.empty_cache()
print('GPU memory after cleanup GB:', round(torch.cuda.memory_allocated() / 1e9, 2))

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    dtype=dtype,
    use_safetensors=True,
).to(device)
if profile['gradient_checkpointing']:
    model.gradient_checkpointing_enable()
    if hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
model.train()
print('Reloaded from', CHECKPOINT_DIR)
print('GPU memory after reload GB:', round(torch.cuda.memory_allocated() / 1e9, 2))

## Step 14 - production CLI canary

Once the lab path works, run the production trainer path. Start with 50 steps so it matches the hands-on canary, then remove `--max-steps 50` for the full config.

In [ ]:
# Uncomment WANDB_MODE if you do not want to upload the run live.
# %env WANDB_MODE=offline

!python -m finpost.training.train --config experiments/colab_qwen_500m.yaml --device cuda --max-steps 50

## Step 15 - production CLI full run

Use this after the 50-step canary writes a checkpoint and shows healthy loss/throughput logging.

In [ ]:
# !python -m finpost.training.train --config experiments/colab_qwen_500m.yaml --device cuda

## Step 16 - optional: plot production metrics from Weights & Biases

This reads the online W&B run. If you trained offline, run `!wandb sync wandb/offline-run-*` first.

In [ ]:
import wandb

PROJECT = 'finpost-phase1'
RUN_NAME = 'qwen2.5-0.5b-colab-demo'

api = wandb.Api()
runs = api.runs(PROJECT, filters={'display_name': RUN_NAME})
if not runs:
    raise RuntimeError(
        f'No W&B run named {RUN_NAME!r} in project {PROJECT!r}. '
        'If you ran offline, sync the offline run first with wandb sync.'
    )
run = runs[0]
history = run.history(keys=['train/loss', 'val/loss', 'train/tokens_per_sec'], pandas=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if 'train/loss' in history:
    axes[0].plot(history['_step'], history['train/loss'], label='train/loss')
if 'val/loss' in history:
    val = history.dropna(subset=['val/loss'])
    axes[0].plot(val['_step'], val['val/loss'], marker='o', label='val/loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('step')
axes[0].legend()
axes[0].grid(alpha=0.3)

if 'train/tokens_per_sec' in history:
    tps = history.dropna(subset=['train/tokens_per_sec'])
    axes[1].plot(tps['_step'], tps['train/tokens_per_sec'], marker='o')
axes[1].set_title('Throughput')
axes[1].set_xlabel('step')
axes[1].set_ylabel('tokens/sec')
axes[1].grid(alpha=0.3)
plt.show()

## Next steps

If the lab path and production 50-step canary both look healthy:

1. Run the production Qwen config without `--max-steps 50`.
2. Open `notebooks/sft_phase1_multitask.ipynb` for the combined vs specialist transfer-learning comparison.
3. Treat Gemma 3 1B as a memory stress test, not the Phase 1 headline model, unless you promote gradient checkpointing and 8-bit AdamW into first-class trainer config fields.